# Constraint Satisfaction Problem

We will practice solving a CSP using backtracking search.
The basic search process will be implemented recursively, and the function will keep track of how many times it's called until a solution is found.
We will then implement the various variable selection strategies discussed in lecture, and check how much of a reduction in search we get from those.

The task at hand is sudoku. The input is a partially completed sudoku puzzle (which is just a 9x9 array with numbers 1-9. Unassigned squares are marked with 0)
Your task is to implement backtracking search, then to enhance it to implement variable selection methods as discussed in lecture.

For input puzzles, have a look here:
 https://www.sudoku-puzzles-online.com/index.php

### Credits
Original code written by Francisco Estrada, Jan. 2020  
Adopted by Bryan Chan, Nov. 2024

In [89]:
import numpy as np

from typing import Tuple

## TODO

Implement the order heuristic discussed in lecture:
1. Choose first the variable with fewest remaining values
2. If (1) has a tie, choose variable that intervenes in more active constraints
3. Once you have chosen a variable, select the least constraining value to try first!

You have to decide how to keep track of 
- Number of valid values left for each unassigned cell
- Number of active constraints for each unassigned cell

And you need to think about how to check which values are least constraining.
Since there may be many, you'll need to sort them from least constraining to most constraining!

BUT! the goal is not to make the most efficient data structure for these things!
Here, we only care about *reduction in the number of calls to `solve`* which tells us something important about the size of the search tree! 

It is assumed you can then make your data structures fast in a language designed to do so, and the improved algorithm (which uses the variable selection process we just described) will beat the 'vanilla' version because of the reduction in the number of nodes that need to be expanded to reach the full solution.

So - don't spend time writing clever data structures, just get this to work, and
see what you can tell about how much it saves in terms of calls to `solve`.

In [90]:
def solve_sudoku(puzzle: np.ndarray, strategy: str = "default") -> Tuple[np.ndarray, int]:
    """
    Solves a Sudoku puzzle and keeps track of call count.

    Args:
    - puzzle (ndarray): a 9x9 array containing the initial puzzle
    - strategy (str): the strategy to choose unassigned variable
    """
    assert strategy in [
        "default",
        "order",
    ]

    if strategy == "default":
        def get_expanded_idxes(state):
            """
            Simply gets the first unassigned variable.

            Args:
            - state (ndarray): a 9x9 array containing the partially-solved puzzle
            """
            
            unassigned_mask = state == 0
            unassigned_idxes = np.where(unassigned_mask)

            unassigned_row_idx, unassigned_col_idx = unassigned_idxes[0][0], unassigned_idxes[1][0]
            return unassigned_row_idx, unassigned_col_idx
    elif strategy == "order":
        def get_expanded_idxes(state):
            """
            Gets the unassigned variable based on the variable-ordering criteria discussed in lecture.

            Args:
            - state (ndarray): a 9x9 array containing the partially-solved puzzle
            """
            # TODO: Modify this to follow steps (1) and (2) of the order-heuristic
            possibleNums = set((1,2,3,4,5,6,7,8,9))
            colConstraints = {0:set(),1:set(),2:set(),3:set(),4:set(),5:set(),6:set(),7:set(),8:set()}
            rowConstraints = {0:set(),1:set(),2:set(),3:set(),4:set(),5:set(),6:set(),7:set(),8:set()}
            subGridConstraints = {0:set(),1:set(),2:set(),3:set(),4:set(),5:set(),6:set(),7:set(),8:set()}
            import heapq
            for indOne, row in enumerate(state):
                for indTwo, cell in enumerate(row):
                    if not cell == 0:
                        subGridConstraints[(indOne//3) * 3 + indTwo//3].add(cell)
                        colConstraints[indTwo].add(cell)
                        rowConstraints[indOne].add(cell)
            unassigned_mask = state == 0
            unassigned_idxes = np.where(unassigned_mask)
            numChoices = {0:[],1:[],2:[],3:[],4:[],5:[],6:[],7:[],8:[],9:[]}
            for i in range(len(unassigned_idxes[0])):
                indOne = unassigned_idxes[0][i]
                indTwo = unassigned_idxes[1][i]
                tempNums = possibleNums.copy()
                tempNums.discard(rowConstraints[indOne])
                tempNums.discard(colConstraints[indTwo])
                tempNums.discard(subGridConstraints[(indOne//3) * 3 + indTwo//3])
                heapq.heappush(numChoices[len(tempNums)],(-(len(rowConstraints[indOne])+len(colConstraints[indTwo])),(indOne,indTwo)))
            for i in range(10):
                if not numChoices[i] == []:
                    return heapq.heappop(numChoices[i])[1]
            return None, None
    else:
        raise ValueError("strategy {} is not supported".format(strategy))

    def solve(state: np.ndarray, call_count: int):
        call_count += 1

        # Check for unassigned variables
        unassigned_mask = state == 0
        if not np.any(unassigned_mask):
            return np.copy(state), call_count

        # NOTE: This chooses the node to expand
        unassigned_row_idx, unassigned_col_idx = get_expanded_idxes(np.copy(state))

        # Get subgrid
        subgrid_row, subgrid_col = (unassigned_row_idx // 3), (unassigned_col_idx // 3)
        subgrid = state[subgrid_row * 3:(subgrid_row + 1) * 3, subgrid_col * 3:(subgrid_col + 1) * 3]

        # Get row
        row = state[unassigned_row_idx, :]

        # Get column
        col = state[:, unassigned_col_idx]

        assigned_values = np.unique(np.concatenate((subgrid.flatten(), row, col)))
        remaining_values = np.setdiff1d(np.arange(10), assigned_values)

        if strategy == "default":
            # NOTE: You can make a generator for this so we can combine the different strategies.
            for remaining_value in remaining_values:
                copied_state = np.copy(state)
                copied_state[unassigned_row_idx, unassigned_col_idx] = remaining_value
                new_solution, call_count = solve(copied_state, call_count)
                if new_solution is not None:
                    return new_solution, call_count
        elif strategy == "order":
            # TODO: Modify this to follow step (3) of the order-heuristic
            occurences = np.zeros((10))

            directions = [(0,1),(1,0),(0,-1),(-1,0)]
            for direction in directions:
                x,y = direction
                nextX, nextY = (unassigned_row_idx + x, unassigned_col_idx + y)
                if not (nextX < 0 or nextY < 0 or nextX >8 or nextY >8):
                    subgrid_row2, subgrid_col2 = (nextX // 3), (nextY // 3)
                    subgrid2 = state[subgrid_row2 * 3:(subgrid_row2 + 1) * 3, subgrid_col2 * 3:(subgrid_col2 + 1) * 3]

                    # Get row
                    row = state[nextX, :]

                    # Get column
                    col = state[:, nextY]

                    assigned = np.unique(np.concatenate((subgrid2.flatten(), row, col)))
                    remaining = np.setdiff1d(np.arange(10), assigned)
                    for remaining_value in remaining:
                        occurences[remaining] += 1
            sorted_remaining_values = sorted(remaining_values, key=lambda val: occurences[val])

            for remaining_value in sorted_remaining_values:
                copied_state = np.copy(state)
                copied_state[unassigned_row_idx, unassigned_col_idx] = remaining_value
                new_solution, call_count = solve(copied_state, call_count)
                if new_solution is not None:
                    return new_solution, call_count

        else:
            raise ValueError("strategy {} is not supported".format(strategy))

        return None, call_count

    return solve(puzzle, 0)

In [91]:
example_puzzle = [
    [0, 3, 2, 0, 6, 0, 0, 8, 9,],
    [0, 0, 0, 0, 1, 0, 0, 5, 0,],
    [0, 6, 0, 0, 0, 0, 7, 0, 0,],
    [0, 0, 7, 1, 0, 0, 0, 0, 5,],
    [0, 0, 0, 0, 0, 3, 0, 0, 2,],
    [0, 9, 0, 0, 0, 0, 3, 0, 7,],
    [1, 0, 0, 9, 0, 5, 0, 0, 4,],
    [2, 0, 0, 0, 0, 0, 0, 9, 0,],
    [0, 4, 0, 0, 3, 0, 0, 0, 0,],
]

example_puzzle = np.array(example_puzzle)

In [92]:
strategy = "default"
strategy = "order" # NOTE: Uncomment this once it's implemented

In [93]:
solution, call_count = solve_sudoku(example_puzzle, strategy)

In [87]:
solution

array([[5, 3, 2, 4, 6, 7, 1, 8, 9],
       [7, 8, 9, 3, 1, 2, 4, 5, 6],
       [4, 6, 1, 8, 5, 9, 7, 2, 3],
       [3, 2, 7, 1, 9, 4, 8, 6, 5],
       [8, 1, 5, 6, 7, 3, 9, 4, 2],
       [6, 9, 4, 5, 2, 8, 3, 1, 7],
       [1, 7, 6, 9, 8, 5, 2, 3, 4],
       [2, 5, 3, 7, 4, 1, 6, 9, 8],
       [9, 4, 8, 2, 3, 6, 5, 7, 1]])

In [94]:
call_count

509